# 10. 공용 Dataset Loader와 사용법 정리

이후 장에서 그대로 사용할 PyTorch Dataset/DataLoader 사용법을 정리합니다.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "synthetic_metal_utils.py").exists():
    search_roots = [Path.cwd(), *Path.cwd().parents]
    search_patterns = ["synthetic_metal_utils.py", "*/synthetic_metal_utils.py", "*/*/synthetic_metal_utils.py"]
    for root in search_roots:
        for pattern in search_patterns:
            matches = list(root.glob(pattern))
            if matches:
                NOTEBOOK_DIR = matches[0].parent
                break
        if (NOTEBOOK_DIR / "synthetic_metal_utils.py").exists():
            break

sys.path.append(str(NOTEBOOK_DIR))
DATA_ROOT = NOTEBOOK_DIR / "data" / "synthetic_metal_seg"

from synthetic_metal_utils import *
set_korean_font()
set_seed(7)

print("NOTEBOOK_DIR:", NOTEBOOK_DIR)
print("DATA_ROOT:", DATA_ROOT)

## Dataset loader

SyntheticMetalDataset은 image tensor, mask tensor, metadata를 반환할 수 있습니다.

In [ ]:
if not (DATA_ROOT / "metadata" / "samples.csv").exists():
    create_synthetic_metal_dataset(DATA_ROOT, n_train=120, n_eval_per_split=24, overwrite=True)

train_ds = SyntheticMetalDataset(DATA_ROOT, "train", return_metadata=True)
val_light_ds = SyntheticMetalDataset(DATA_ROOT, "val_unseen_light", return_metadata=True)
len(train_ds), len(val_light_ds)

## 한 sample 확인

image는 CxHxW float tensor, mask는 HxW long tensor입니다.

In [ ]:
image_t, mask_t, meta = train_ds[0]
image_t.shape, image_t.dtype, mask_t.shape, mask_t.dtype, {k: meta[k] for k in ["split", "domain_id", "light_r", "light_g", "light_b"]}

## DataLoader smoke test

torch가 설치되어 있으면 batch loading까지 확인합니다.

In [ ]:
try:
    from torch.utils.data import DataLoader
    loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    images, masks, metas = next(iter(loader))
    print(images.shape, masks.shape)
except Exception as exc:
    print("DataLoader 확인을 건너뜁니다:", exc)

## 이후 장에서 사용할 split

2장 이후에는 같은 DATA_ROOT를 사용해 split별 성능을 비교합니다.

In [ ]:
list_splits(DATA_ROOT)